# 8. Probabilistic forecasting

Point forecasts tell you **what** the price will be; probabilistic forecasts
tell you **how wrong you might be**, and that second piece of information is
what a battery operator actually trades on.

This notebook combines quantile forecasts from LEAR (NB 06) and LightGBM
(NB 07) using **quantile regression averaging** (QRA), calibrates the
resulting intervals with **conformal prediction**, and evaluates the full
probabilistic forecast with proper scoring rules, reliability diagrams,
and -- most importantly -- its impact on battery dispatch revenue.

## Objectives

- Combine quantile forecasts from two models using QRA and inspect the weights.
- Assess calibration via reliability diagrams and apply conformal prediction.
- Measure the sharpness--coverage trade-off.
- Test statistical significance with the Diebold--Mariano test.
- Evaluate spike coverage.
- Run calibrated forecasts through MPC dispatch and compare capture ratios.
- Produce an updated scorecard of all models.

## Prerequisites

- Processed data at `data/processed/SA1_30min.parquet` (NB 01).
- Familiarity with LEAR (NB 06) and GBT quantile (NB 07) models.
- `grian` library installed (`pip install -e .`).

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from grian.config import load_config, repo_root
from grian.dispatch import capture_ratio, schedule
from grian.features import build_matrix
from grian.metrics import crps, mae
from grian.models.conformal import ConformalWrapper
from grian.models.gbt import GBTQuantile
from grian.models.lear import LEAR
from grian.models.qra import QRA
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
np.random.seed(cfg["seed"])

QUANTILES = cfg["quantiles"]
print(f"Region: {cfg['region']}")
print(f"Quantile levels: {QUANTILES}")

---

## Load and split data

In [ ]:
df = pd.read_parquet(repo_root() / "data" / "processed" / "SA1_30min.parquet")
print(f"Full dataset: {df.shape[0]:,} rows, {df.index.min()} to {df.index.max()}")
df.head(3)

In [ ]:
# Build feature matrix
X = build_matrix(df[["price"]], df[["demand"]])
y_raw = df.loc[X.index, "price"]
y = np.arcsinh(y_raw)

print(f"Feature matrix: {X.shape}")
print(f"Target range (arcsinh): [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Three-way split: train (pre-calibration) / calibration / test
cal_start = "2023-04-01"

X_precal = X.loc[:"2023-03-31"]
y_precal = y.loc[X_precal.index]

X_cal = X.loc[cal_start:cfg["train_end"]]
y_cal = y.loc[X_cal.index]
y_cal_raw = y_raw.loc[X_cal.index]

X_test = X.loc[cfg["test_start"]:cfg["test_end"]]
y_test = y.loc[X_test.index]
y_test_raw = y_raw.loc[X_test.index]

print(f"Pre-calibration: {len(X_precal):,} rows")
print(f"Calibration:     {len(X_cal):,} rows ({X_cal.index.min().date()} to {X_cal.index.max().date()})")
print(f"Test:            {len(X_test):,} rows ({X_test.index.min().date()} to {X_test.index.max().date()})")

---

## Train base models

We retrain LEAR and GBT from scratch on the pre-calibration period, then
generate quantile forecasts on both the calibration and test sets.

In [ ]:
# --- LEAR (point forecast, replicated across quantiles) ---
lear = LEAR(seed=cfg["seed"])
lear.fit(X_precal, y_precal)

lear_cal_point = lear.predict(X_cal).values
lear_test_point = lear.predict(X_test).values

# LEAR is a point model -- broadcast to all quantile columns.
# QRA will learn how to spread these into a distribution.
lear_cal_q = np.tile(lear_cal_point[:, None], (1, len(QUANTILES)))
lear_test_q = np.tile(lear_test_point[:, None], (1, len(QUANTILES)))

print(f"LEAR point forecast -- cal MAE (arcsinh): {mae(y_cal, lear_cal_point):.4f}")

In [ ]:
# --- GBT quantile model ---
gbt = GBTQuantile(quantiles=QUANTILES, seed=cfg["seed"])
gbt.fit(X_precal, y_precal)

gbt_cal_df = gbt.predict(X_cal)
gbt_test_df = gbt.predict(X_test)

gbt_cal_q = gbt_cal_df.values
gbt_test_q = gbt_test_df.values

print(f"GBT median forecast -- cal MAE (arcsinh): {mae(y_cal, gbt_cal_q[:, 3]):.4f}")
print(f"GBT CRPS (arcsinh): {crps(y_cal.values, gbt_cal_q, QUANTILES):.4f}")

Let's look at the raw quantile fan from GBT on a short window before we
start combining.

In [ ]:
# Quick visual: GBT quantile fan on a slice of calibration data
slice_idx = X_cal.index[:48*3]  # 3 days
sl = slice(0, len(slice_idx))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(slice_idx, np.sinh(y_cal.values[sl]), color="black", lw=1.5, label="Actual")
ax.plot(slice_idx, np.sinh(gbt_cal_q[sl, 3]), color="tab:blue", lw=1, label="GBT median")
ax.fill_between(slice_idx, np.sinh(gbt_cal_q[sl, 0]), np.sinh(gbt_cal_q[sl, -1]),
                alpha=0.15, color="tab:blue", label="5--95% PI")
ax.fill_between(slice_idx, np.sinh(gbt_cal_q[sl, 2]), np.sinh(gbt_cal_q[sl, 4]),
                alpha=0.25, color="tab:blue", label="25--75% PI")
ax.set(ylabel="Price ($/MWh)", title="GBT quantile fan -- calibration sample")
ax.legend(loc="upper right")
plt.tight_layout()
save_fig(fig, "08_gbt_quantile_fan_sample")
plt.show()

---

## 1. Quantile combination with QRA

Quantile regression averaging (QRA) treats each model's quantile forecast
as a feature and fits a quantile regression at each level.  The result is
an optimally weighted combination that can outperform either input model.

We feed it two sets of quantile forecasts (LEAR and GBT) on the
calibration period, then generate combined forecasts on the test set.

In [ ]:
qra = QRA(quantiles=QUANTILES, alpha=0.0)
qra.fit(
    forecasts={"LEAR": lear_cal_q, "GBT": gbt_cal_q},
    actual=y_cal.values,
)

# Combined forecasts on test set
qra_test_q = qra.predict({"LEAR": lear_test_q, "GBT": gbt_test_q})

print(f"QRA combined CRPS (arcsinh): {crps(y_test.values, qra_test_q, QUANTILES):.4f}")
print(f"GBT solo CRPS (arcsinh):     {crps(y_test.values, gbt_test_q, QUANTILES):.4f}")

### QRA weights: which model dominates at each quantile?

In [ ]:
weights = qra.weights()
weight_df = pd.DataFrame(
    {f"q{QUANTILES[qi]:.2f}": w for qi, w in weights.items()}
).T
weight_df.index.name = "quantile"
print("QRA combination weights (per quantile level):")
weight_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
weight_df.plot(kind="bar", ax=ax)
ax.set(ylabel="Weight", xlabel="Quantile level",
       title="QRA weights by quantile level")
ax.axhline(0, color="black", lw=0.8)
ax.legend(title="Model")
plt.tight_layout()
save_fig(fig, "08_qra_weights")
plt.show()

The weight chart reveals which model the combiner trusts at each risk
level.  Typically the GBT model -- with its native quantile objective --
dominates the tails, while LEAR's linear structure contributes more to
the central quantiles.

---

## 2. Calibration check: reliability diagram

A well-calibrated probabilistic forecast has the property that the
fraction of observations falling below each predicted quantile matches
the nominal level.  The **reliability diagram** plots observed coverage
against nominal coverage -- a perfectly calibrated model lies on the
diagonal.

In [ ]:
def observed_coverage(actual, quantile_forecasts, quantiles):
    """Compute observed coverage for each quantile level."""
    actual = np.asarray(actual)
    qf = np.asarray(quantile_forecasts)
    coverages = []
    for i, q in enumerate(quantiles):
        coverages.append(np.mean(actual <= qf[:, i]))
    return np.array(coverages)


def plot_reliability(coverages_dict, quantiles, ax=None, title="Reliability diagram"):
    """Plot reliability diagram for one or more forecast sets."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    for name, cov in coverages_dict.items():
        ax.plot(quantiles, cov, "o-", markersize=6, label=name)
    ax.set(xlabel="Nominal coverage", ylabel="Observed coverage", title=title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.legend()
    return ax

In [ ]:
# Observed coverage of raw models on the test set
cov_gbt = observed_coverage(y_test.values, gbt_test_q, QUANTILES)
cov_qra = observed_coverage(y_test.values, qra_test_q, QUANTILES)

fig, ax = plt.subplots(figsize=(6, 6))
plot_reliability(
    {"GBT (raw)": cov_gbt, "QRA combined": cov_qra},
    QUANTILES, ax=ax,
    title="Reliability diagram -- raw quantile forecasts"
)
plt.tight_layout()
save_fig(fig, "08_reliability_raw")
plt.show()

# Print coverage table
cov_table = pd.DataFrame({
    "Nominal": QUANTILES,
    "GBT observed": cov_gbt,
    "QRA observed": cov_qra,
}).set_index("Nominal")
cov_table

Deviations from the diagonal indicate miscalibration.  Upper quantiles
above the diagonal mean the model's upper bounds are too wide (over-covers);
lower quantiles below the diagonal mean the lower bounds are too tight
(under-covers).

---

## 3. Conformal prediction

Conformal prediction provides a **distribution-free coverage guarantee**.
It uses a held-out calibration set to compute conformity scores and
adjusts the quantile widths so that empirical coverage meets or exceeds
the nominal level -- without any distributional assumptions.

We calibrate on the QRA forecasts from the calibration period, then
adjust the test set forecasts.

In [ ]:
# Generate QRA forecasts on the calibration set for conformal calibration
qra_cal_q = qra.predict({"LEAR": lear_cal_q, "GBT": gbt_cal_q})

# Fit conformal wrapper
conformal = ConformalWrapper(quantiles=QUANTILES)
conformal.calibrate(qra_cal_q, y_cal.values)

print("Conformal adjustments per quantile:")
for i, q in enumerate(QUANTILES):
    direction = "down" if q <= 0.5 else "up"
    print(f"  q{q:.2f}: shift {direction} by {conformal.adjustments_[i]:.4f}")

In [ ]:
# Adjust test set forecasts
qra_conf_test_q = conformal.adjust(qra_test_q)

# Coverage after conformal adjustment
cov_conf = observed_coverage(y_test.values, qra_conf_test_q, QUANTILES)

fig, ax = plt.subplots(figsize=(6, 6))
plot_reliability(
    {"QRA (raw)": cov_qra, "QRA + conformal": cov_conf},
    QUANTILES, ax=ax,
    title="Reliability diagram -- before and after conformal adjustment"
)
plt.tight_layout()
save_fig(fig, "08_reliability_conformal")
plt.show()

cov_table_conf = pd.DataFrame({
    "Nominal": QUANTILES,
    "QRA raw": cov_qra,
    "QRA + conformal": cov_conf,
}).set_index("Nominal")
cov_table_conf

After conformal adjustment the observed coverage should hug the diagonal
more closely.  The price of coverage is wider intervals -- which we
quantify next.

---

## 4. Sharpness vs coverage trade-off

A forecast that covers everything is useless if the intervals span the
entire price range.  **Sharpness** measures how narrow the prediction
intervals are; the goal is maximal sharpness subject to calibration.

In [ ]:
def interval_width(qf, lower_idx, upper_idx):
    """Mean interval width between two quantile levels (in original $/MWh)."""
    return np.mean(np.sinh(qf[:, upper_idx]) - np.sinh(qf[:, lower_idx]))

def interval_coverage(actual, qf, lower_idx, upper_idx):
    """Fraction of actuals within the prediction interval."""
    return np.mean((actual >= qf[:, lower_idx]) & (actual <= qf[:, upper_idx]))

# Define symmetric intervals: 90% (q0.05-q0.95), 80% (q0.10-q0.90), 50% (q0.25-q0.75)
intervals = [
    ("90% PI", 0, 6),  # q0.05 -- q0.95
    ("80% PI", 1, 5),  # q0.10 -- q0.90
    ("50% PI", 2, 4),  # q0.25 -- q0.75
]

rows = []
for name, lo, hi in intervals:
    nominal = QUANTILES[hi] - QUANTILES[lo]
    for label, qf in [("GBT", gbt_test_q), ("QRA", qra_test_q), ("QRA+conf", qra_conf_test_q)]:
        width = interval_width(qf, lo, hi)
        cov = interval_coverage(y_test.values, qf, lo, hi)
        rows.append({"Interval": name, "Model": label, "Nominal": nominal,
                     "Coverage": cov, "Width ($/MWh)": width})

sharpness_df = pd.DataFrame(rows)
sharpness_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: width comparison
for name, grp in sharpness_df.groupby("Model"):
    axes[0].plot(grp["Nominal"], grp["Width ($/MWh)"], "o-", label=name)
axes[0].set(xlabel="Nominal coverage", ylabel="Mean interval width ($/MWh)",
            title="Sharpness: narrower is better")
axes[0].legend()

# Right: coverage vs nominal
for name, grp in sharpness_df.groupby("Model"):
    axes[1].plot(grp["Nominal"], grp["Coverage"], "o-", label=name)
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set(xlabel="Nominal coverage", ylabel="Empirical coverage",
            title="Coverage: closer to diagonal is better")
axes[1].legend()

plt.tight_layout()
save_fig(fig, "08_sharpness_coverage")
plt.show()

The conformal wrapper widens intervals to guarantee coverage.  The key
question is whether the widening is modest enough to remain useful for
dispatch decisions.

---

## 5. Diebold--Mariano test

Is the combined QRA forecast **statistically significantly** better than
each individual model?  The Diebold--Mariano (DM) test compares the
predictive accuracy of two forecasts using a loss differential series.

In [ ]:
def dm_test(actual, forecast_1, forecast_2, quantiles, h=1):
    """Diebold-Mariano test on mean pinball loss.
    
    H0: forecast_1 and forecast_2 have equal predictive accuracy.
    Negative test statistic means forecast_1 is better.
    """
    actual = np.asarray(actual)
    # Compute per-observation average pinball loss across quantiles
    loss_1 = np.zeros(len(actual))
    loss_2 = np.zeros(len(actual))
    for i, q in enumerate(quantiles):
        diff_1 = actual - forecast_1[:, i]
        diff_2 = actual - forecast_2[:, i]
        loss_1 += np.where(diff_1 >= 0, q * diff_1, (q - 1) * diff_1)
        loss_2 += np.where(diff_2 >= 0, q * diff_2, (q - 1) * diff_2)
    loss_1 /= len(quantiles)
    loss_2 /= len(quantiles)
    
    d = loss_1 - loss_2
    d_mean = np.mean(d)
    
    # HAC variance estimate (Newey-West with h-1 lags)
    n = len(d)
    gamma_0 = np.var(d, ddof=1)
    d_centered = d - d_mean
    hac_var = gamma_0
    for k in range(1, h):
        gamma_k = np.sum(d_centered[k:] * d_centered[:-k]) / n
        hac_var += 2 * (1 - k / h) * gamma_k
    
    dm_stat = d_mean / np.sqrt(hac_var / n)
    p_value = 2 * stats.norm.sf(np.abs(dm_stat))
    
    return dm_stat, p_value

In [ ]:
# QRA vs GBT
dm_qra_gbt, p_qra_gbt = dm_test(y_test.values, qra_test_q, gbt_test_q, QUANTILES)
print("DM test: QRA vs GBT")
print(f"  Statistic: {dm_qra_gbt:.3f}")
print(f"  p-value:   {p_qra_gbt:.4f}")
print(f"  {'QRA significantly better' if dm_qra_gbt < 0 and p_qra_gbt < 0.05 else 'No significant difference' if p_qra_gbt >= 0.05 else 'GBT significantly better'}")
print()

# QRA vs LEAR (broadcast as quantiles)
dm_qra_lear, p_qra_lear = dm_test(y_test.values, qra_test_q, lear_test_q, QUANTILES)
print("DM test: QRA vs LEAR")
print(f"  Statistic: {dm_qra_lear:.3f}")
print(f"  p-value:   {p_qra_lear:.4f}")
print(f"  {'QRA significantly better' if dm_qra_lear < 0 and p_qra_lear < 0.05 else 'No significant difference' if p_qra_lear >= 0.05 else 'LEAR significantly better'}")

---

## 6. Spike coverage

Extreme prices (above $300/MWh) drive most of a battery's revenue.
A probabilistic forecast is only useful if it captures these spikes
within its prediction intervals.

In [ ]:
spike_threshold = cfg["spike_threshold_aud"]
spike_mask = y_test_raw.values > spike_threshold
n_spikes = spike_mask.sum()
print(f"Spike threshold: ${spike_threshold}/MWh")
print(f"Test set spikes: {n_spikes} ({100*n_spikes/len(y_test_raw):.1f}% of periods)")

In [ ]:
# Coverage of spikes: what fraction of spikes fall within prediction intervals?
if n_spikes > 0:
    spike_rows = []
    for name, qf in [("GBT", gbt_test_q), ("QRA", qra_test_q), ("QRA+conf", qra_conf_test_q)]:
        for interval_name, lo, hi in intervals:
            # Convert back to $/MWh for comparison with raw prices
            lower = np.sinh(qf[spike_mask, lo])
            upper = np.sinh(qf[spike_mask, hi])
            actual_spikes = y_test_raw.values[spike_mask]
            covered = np.mean((actual_spikes >= lower) & (actual_spikes <= upper))
            spike_rows.append({"Model": name, "Interval": interval_name,
                               "Spike coverage": covered})
    
    spike_cov_df = pd.DataFrame(spike_rows)
    spike_cov_pivot = spike_cov_df.pivot(index="Model", columns="Interval", values="Spike coverage")
    print("Spike coverage (fraction of spikes within prediction intervals):")
    display(spike_cov_pivot)
else:
    print("No spikes in test period -- skipping spike coverage analysis.")

In [ ]:
# Visualise: prediction intervals around spikes
if n_spikes > 0:
    # Find a window with spikes
    spike_idx = np.where(spike_mask)[0]
    center = spike_idx[len(spike_idx) // 2]
    window = slice(max(0, center - 48), min(len(y_test), center + 48))
    win_idx = X_test.index[window]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(win_idx, y_test_raw.values[window], color="black", lw=1.5, label="Actual")
    ax.plot(win_idx, np.sinh(qra_conf_test_q[window, 3]), color="tab:orange", lw=1,
            label="QRA+conf median")
    ax.fill_between(win_idx,
                    np.sinh(qra_conf_test_q[window, 0]),
                    np.sinh(qra_conf_test_q[window, -1]),
                    alpha=0.15, color="tab:orange", label="5--95% PI")
    ax.axhline(spike_threshold, color="red", ls=":", lw=1, label=f"Spike threshold (${spike_threshold})")
    ax.set(ylabel="Price ($/MWh)", title="QRA + conformal intervals around price spikes")
    ax.legend(loc="upper right")
    plt.tight_layout()
    save_fig(fig, "08_spike_coverage")
    plt.show()

---

## 7. Dispatch: probabilistic forecast to battery revenue

We compare two dispatch strategies:

1. **Certainty-equivalent MPC**: use the median (q0.50) as a point
   forecast and schedule against it.
2. **Scenario MPC**: sample scenarios from the quantile distribution,
   solve the LP for each, and average the first-period actions.

Both use rolling MPC: at each step we re-forecast and re-optimise.

In [ ]:
battery_kwargs = {
    "power_mw": cfg["battery"]["power_mw"],
    "duration_hours": cfg["battery"]["duration_hours"],
    "efficiency": cfg["battery"]["efficiency_roundtrip"],
    "max_cycles": cfg["battery"]["max_cycles_per_day"],
}
horizon = cfg["horizon_periods"]
print(f"Battery: {battery_kwargs}")
print(f"Horizon: {horizon} periods")

In [ ]:
# Perfect foresight benchmark
actual_prices = y_test_raw.values
perfect = schedule(actual_prices, **battery_kwargs)
perfect_revenue = perfect["revenue"]
print(f"Perfect foresight revenue: ${perfect_revenue:,.0f}")

In [ ]:
def daily_dispatch(price_forecasts, actual_prices, battery_kwargs):
    """Run daily dispatch and accumulate revenue.
    
    Dispatches day-by-day (48 periods), using the forecast for each day
    and scoring against actual prices.
    """
    T = len(actual_prices)
    dt = 0.5
    total_revenue = 0.0
    daily_revenues = []
    
    for start in range(0, T - 47, 48):  # step by day
        end = start + 48
        forecast_day = price_forecasts[start:end]
        actual_day = actual_prices[start:end]
        
        result = schedule(forecast_day, **battery_kwargs)
        if result["status"] == "optimal":
            # Revenue scored against actual prices
            net_action = result["discharge"] - result["charge"]
            day_revenue = float(np.sum(actual_day * net_action * dt))
        else:
            day_revenue = 0.0
        
        total_revenue += day_revenue
        daily_revenues.append(day_revenue)
    
    return total_revenue, daily_revenues

In [ ]:
# Strategy 1: Certainty-equivalent (median forecast)
median_forecast = np.sinh(qra_conf_test_q[:, 3])  # q0.50, inverted
rev_median, daily_rev_median = daily_dispatch(median_forecast, actual_prices, battery_kwargs)
cr_median = capture_ratio(rev_median, perfect_revenue)
print(f"Median MPC revenue: ${rev_median:,.0f}  |  Capture ratio: {cr_median:.3f}")

In [ ]:
# Strategy 2: Scenario MPC
# Sample scenarios by interpolating between quantile levels
n_scenarios = 20
rng = np.random.RandomState(cfg["seed"])

def sample_scenarios(quantile_forecasts, quantiles, n_scenarios, rng):
    """Sample price scenarios by interpolating the quantile function."""
    T, nq = quantile_forecasts.shape
    scenarios = np.empty((n_scenarios, T))
    for s in range(n_scenarios):
        u = rng.uniform(0, 1, T)
        for t in range(T):
            scenarios[s, t] = np.interp(u[t], quantiles, quantile_forecasts[t, :])
    return scenarios


def scenario_dispatch(quantile_forecasts, quantiles, actual_prices,
                      battery_kwargs, n_scenarios=20, rng=None):
    """Scenario-based MPC: average optimal schedules across scenarios."""
    T = len(actual_prices)
    dt = 0.5
    total_revenue = 0.0
    daily_revenues = []
    
    for start in range(0, T - 47, 48):
        end = start + 48
        qf_day = quantile_forecasts[start:end, :]
        actual_day = actual_prices[start:end]
        
        # Sample scenarios in original price space
        scenarios = sample_scenarios(qf_day, quantiles, n_scenarios, rng)
        scenarios = np.sinh(scenarios)  # invert arcsinh
        
        # Solve LP for each scenario, average the actions
        avg_charge = np.zeros(48)
        avg_discharge = np.zeros(48)
        valid = 0
        
        for s in range(n_scenarios):
            result = schedule(scenarios[s], **battery_kwargs)
            if result["status"] == "optimal":
                avg_charge += result["charge"]
                avg_discharge += result["discharge"]
                valid += 1
        
        if valid > 0:
            avg_charge /= valid
            avg_discharge /= valid
            net_action = avg_discharge - avg_charge
            day_revenue = float(np.sum(actual_day * net_action * dt))
        else:
            day_revenue = 0.0
        
        total_revenue += day_revenue
        daily_revenues.append(day_revenue)
    
    return total_revenue, daily_revenues


rev_scenario, daily_rev_scenario = scenario_dispatch(
    qra_conf_test_q, QUANTILES, actual_prices,
    battery_kwargs, n_scenarios=n_scenarios, rng=rng
)
cr_scenario = capture_ratio(rev_scenario, perfect_revenue)
print(f"Scenario MPC revenue: ${rev_scenario:,.0f}  |  Capture ratio: {cr_scenario:.3f}")

In [ ]:
# Also dispatch with individual models for comparison
gbt_median = np.sinh(gbt_test_q[:, 3])  # GBT median
lear_point = np.sinh(lear_test_point)    # LEAR point

rev_gbt, daily_rev_gbt = daily_dispatch(gbt_median, actual_prices, battery_kwargs)
rev_lear, daily_rev_lear = daily_dispatch(lear_point, actual_prices, battery_kwargs)

cr_gbt = capture_ratio(rev_gbt, perfect_revenue)
cr_lear = capture_ratio(rev_lear, perfect_revenue)

print(f"LEAR point MPC:    ${rev_lear:,.0f}  |  CR: {cr_lear:.3f}")
print(f"GBT median MPC:    ${rev_gbt:,.0f}  |  CR: {cr_gbt:.3f}")
print(f"QRA median MPC:    ${rev_median:,.0f}  |  CR: {cr_median:.3f}")
print(f"QRA scenario MPC:  ${rev_scenario:,.0f}  |  CR: {cr_scenario:.3f}")

In [ ]:
# Plot cumulative revenue comparison
fig, ax = plt.subplots(figsize=(14, 5))

for label, daily_rev in [("LEAR point", daily_rev_lear),
                          ("GBT median", daily_rev_gbt),
                          ("QRA median", daily_rev_median),
                          ("QRA scenario", daily_rev_scenario)]:
    ax.plot(np.cumsum(daily_rev), label=label)

ax.set(xlabel="Day", ylabel="Cumulative revenue ($)",
       title="Battery dispatch: cumulative revenue by strategy")
ax.legend()
plt.tight_layout()
save_fig(fig, "08_cumulative_revenue")
plt.show()

---

## 8. Full scorecard

All models, all metrics, capture ratio as the lead column.

In [ ]:
# Compute CRPS and point MAE for all variants
y_test_arr = y_test.values
y_test_raw_arr = y_test_raw.values

scorecard_rows = []

# LEAR
lear_mae = mae(y_test_raw_arr, np.sinh(lear_test_point))
lear_crps = crps(y_test_arr, lear_test_q, QUANTILES)
scorecard_rows.append({
    "Model": "LEAR",
    "Capture ratio": cr_lear,
    "Revenue ($)": rev_lear,
    "MAE ($/MWh)": lear_mae,
    "CRPS (arcsinh)": lear_crps,
})

# GBT
gbt_mae = mae(y_test_raw_arr, np.sinh(gbt_test_q[:, 3]))
gbt_crps = crps(y_test_arr, gbt_test_q, QUANTILES)
scorecard_rows.append({
    "Model": "GBT (quantile)",
    "Capture ratio": cr_gbt,
    "Revenue ($)": rev_gbt,
    "MAE ($/MWh)": gbt_mae,
    "CRPS (arcsinh)": gbt_crps,
})

# QRA
qra_mae = mae(y_test_raw_arr, np.sinh(qra_test_q[:, 3]))
qra_crps = crps(y_test_arr, qra_test_q, QUANTILES)
scorecard_rows.append({
    "Model": "QRA combined",
    "Capture ratio": cr_median,
    "Revenue ($)": rev_median,
    "MAE ($/MWh)": qra_mae,
    "CRPS (arcsinh)": qra_crps,
})

# QRA + conformal
conf_mae = mae(y_test_raw_arr, np.sinh(qra_conf_test_q[:, 3]))
conf_crps = crps(y_test_arr, qra_conf_test_q, QUANTILES)
scorecard_rows.append({
    "Model": "QRA + conformal",
    "Capture ratio": cr_median,
    "Revenue ($)": rev_median,
    "MAE ($/MWh)": conf_mae,
    "CRPS (arcsinh)": conf_crps,
})

# QRA scenario dispatch
scorecard_rows.append({
    "Model": "QRA scenario MPC",
    "Capture ratio": cr_scenario,
    "Revenue ($)": rev_scenario,
    "MAE ($/MWh)": conf_mae,
    "CRPS (arcsinh)": conf_crps,
})

scorecard = pd.DataFrame(scorecard_rows).set_index("Model")
scorecard["Capture ratio"] = scorecard["Capture ratio"].map("{:.3f}".format)
scorecard["Revenue ($)"] = scorecard["Revenue ($)"].map("${:,.0f}".format)
scorecard["MAE ($/MWh)"] = scorecard["MAE ($/MWh)"].map("{:.2f}".format)
scorecard["CRPS (arcsinh)"] = scorecard["CRPS (arcsinh)"].map("{:.4f}".format)
scorecard

---

## Exercises

### Exercise 1: Adaptive conformal widening

The conformal wrapper widens all intervals by a fixed amount. But price
volatility varies: quiet overnight periods need narrow intervals while
afternoon peaks need wide ones.

Can you make the conformal widening **adaptive** -- wider during volatile
periods, narrower during calm ones?

<details><summary>Hint 1</summary>
Compute a volatility proxy for each period -- for example, the rolling
standard deviation of recent prices, or the hour-of-day average absolute
return.
</details>

<details><summary>Hint 2</summary>
Instead of a single scalar adjustment, compute the conformity scores
separately for high-volatility and low-volatility subsets of the
calibration data, then apply the appropriate adjustment based on the
current period's volatility regime.
</details>

<details><summary>Solution</summary>

```python
# Adaptive conformal: split calibration data by volatility regime
rolling_vol = y_cal_raw.rolling(48).std()
vol_median = rolling_vol.median()
high_vol = rolling_vol > vol_median

# Calibrate separately for each regime
conf_high = ConformalWrapper(quantiles=QUANTILES)
conf_low = ConformalWrapper(quantiles=QUANTILES)

mask_high = high_vol.loc[X_cal.index].fillna(False).values
mask_low = ~mask_high

conf_high.calibrate(qra_cal_q[mask_high], y_cal.values[mask_high])
conf_low.calibrate(qra_cal_q[mask_low], y_cal.values[mask_low])

# Apply to test set based on test-time volatility
rolling_vol_test = y_test_raw.rolling(48).std().fillna(method="bfill")
test_high = (rolling_vol_test > vol_median).values

adaptive_q = qra_test_q.copy()
adaptive_q[test_high] = conf_high.adjust(qra_test_q[test_high])
adaptive_q[~test_high] = conf_low.adjust(qra_test_q[~test_high])

# Compare coverage
cov_adaptive = observed_coverage(y_test.values, adaptive_q, QUANTILES)
cov_uniform = observed_coverage(y_test.values, qra_conf_test_q, QUANTILES)

print("Coverage comparison:")
print(pd.DataFrame({
    "Nominal": QUANTILES,
    "Uniform conformal": cov_uniform,
    "Adaptive conformal": cov_adaptive,
}).set_index("Nominal"))

# Compare interval widths
width_uniform = np.mean(np.sinh(qra_conf_test_q[:, -1]) - np.sinh(qra_conf_test_q[:, 0]))
width_adaptive = np.mean(np.sinh(adaptive_q[:, -1]) - np.sinh(adaptive_q[:, 0]))
print(f"\n90% PI width -- uniform: ${width_uniform:.1f}, adaptive: ${width_adaptive:.1f}")
```
</details>

In [ ]:
# Your analysis here

### Exercise 2: QRA weight stability

The QRA weights above were fitted on one specific calibration period
(April--June 2023). Do the weights change if you use a different
calibration window? What does this tell you about model stability?

<details><summary>Hint 1</summary>
Try calibrating on the first half vs the second half of the calibration
period, or use a rolling 1-month window and plot how the weights evolve.
</details>

<details><summary>Hint 2</summary>
If the weights change dramatically between periods, the combination is
fragile and you might prefer a simpler equal-weight average.  If they're
stable, QRA is capturing a robust signal about model complementarity.
</details>

<details><summary>Solution</summary>

```python
# Split calibration period in half and compare weights
mid = len(X_cal) // 2

qra_first = QRA(quantiles=QUANTILES)
qra_first.fit(
    {"LEAR": lear_cal_q[:mid], "GBT": gbt_cal_q[:mid]},
    y_cal.values[:mid],
)

qra_second = QRA(quantiles=QUANTILES)
qra_second.fit(
    {"LEAR": lear_cal_q[mid:], "GBT": gbt_cal_q[mid:]},
    y_cal.values[mid:],
)

w_first = pd.DataFrame(
    {f"q{QUANTILES[qi]:.2f}": w for qi, w in qra_first.weights().items()}
).T
w_second = pd.DataFrame(
    {f"q{QUANTILES[qi]:.2f}": w for qi, w in qra_second.weights().items()}
).T

print("First half weights:")
print(w_first)
print("\nSecond half weights:")
print(w_second)
print("\nDifference (second - first):")
print(w_second - w_first)
```
</details>

In [ ]:
# Your analysis here

### Exercise 3: Cumulative revenue advantage

Plot the cumulative revenue *difference* between the combined forecast
MPC and the best individual model MPC.  When does combination help most?

<details><summary>Hint 1</summary>
Compute the daily revenue difference and plot `np.cumsum()`.  Look at
when the curve rises steeply -- those are days where combination adds
the most value.
</details>

<details><summary>Hint 2</summary>
Overlay the actual price volatility (e.g. daily price range) on a
secondary axis. Does the revenue advantage correlate with volatile days?
</details>

<details><summary>Solution</summary>

```python
# Determine the best individual model
best_individual = daily_rev_gbt if rev_gbt > rev_lear else daily_rev_lear
best_name = "GBT" if rev_gbt > rev_lear else "LEAR"

# Revenue difference: QRA - best individual
n_days = min(len(daily_rev_median), len(best_individual))
diff_median = np.array(daily_rev_median[:n_days]) - np.array(best_individual[:n_days])
diff_scenario = np.array(daily_rev_scenario[:n_days]) - np.array(best_individual[:n_days])

# Daily price range as volatility proxy
daily_range = []
for start in range(0, len(actual_prices) - 47, 48):
    daily_range.append(actual_prices[start:start+48].max() - actual_prices[start:start+48].min())
daily_range = np.array(daily_range[:n_days])

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(np.cumsum(diff_median), label=f"QRA median - {best_name}", color="tab:blue")
ax1.plot(np.cumsum(diff_scenario), label=f"QRA scenario - {best_name}", color="tab:orange")
ax1.axhline(0, color="black", lw=0.8, ls="--")
ax1.set(xlabel="Day", ylabel="Cumulative revenue difference ($)")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.fill_between(range(n_days), daily_range, alpha=0.15, color="gray", label="Daily price range")
ax2.set_ylabel("Daily price range ($/MWh)")
ax2.legend(loc="upper right")

ax1.set_title(f"Revenue advantage of QRA over {best_name}")
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Your analysis here

---

## Summary

1. **QRA combination** optimally weights LEAR and GBT quantile forecasts.
   The weight chart shows which model the combiner trusts at each quantile
   level -- typically the tree model dominates the tails.
2. **Raw quantiles are miscalibrated**: the reliability diagram shows
   systematic under- or over-coverage at several levels.
3. **Conformal prediction** corrects the coverage to match nominal levels
   at the cost of wider intervals.
4. **Sharpness vs coverage**: conformal widening is modest for the central
   intervals but more noticeable in the tails.
5. **Statistical significance**: the DM test quantifies whether the
   improvement from combination is robust or within noise.
6. **Spike coverage**: extreme prices are the hardest to bracket. The 90%
   interval captures most spikes, but narrower intervals miss many.
7. **Dispatch value**: the scenario MPC can exploit distributional
   information that the certainty-equivalent approach ignores.
8. **Scorecard**: capture ratio is the bottom line -- the best forecasting
   model is the one that makes the battery the most money.

---

## Report

In [ ]:
import textwrap

report_dir = repo_root() / "outputs" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report = textwrap.dedent(f"""\
    NB08 Probabilistic Forecasting Report
    ======================================
    Region: {cfg['region']}
    Test period: {cfg['test_start']} to {cfg['test_end']}
    Quantile levels: {QUANTILES}

    CRPS (arcsinh scale)
    --------------------
    LEAR (broadcast):  {lear_crps:.4f}
    GBT quantile:      {gbt_crps:.4f}
    QRA combined:       {qra_crps:.4f}
    QRA + conformal:    {conf_crps:.4f}

    Dispatch Revenue
    ----------------
    Perfect foresight:  ${perfect_revenue:,.0f}
    LEAR point MPC:     ${rev_lear:,.0f}  (CR: {cr_lear:.3f})
    GBT median MPC:     ${rev_gbt:,.0f}  (CR: {cr_gbt:.3f})
    QRA median MPC:     ${rev_median:,.0f}  (CR: {cr_median:.3f})
    QRA scenario MPC:   ${rev_scenario:,.0f}  (CR: {cr_scenario:.3f})

    DM Test: QRA vs GBT
    --------------------
    Statistic: {dm_qra_gbt:.3f}, p-value: {p_qra_gbt:.4f}

    Spikes (>{spike_threshold} $/MWh): {n_spikes} in test set
""")

report_path = report_dir / "08_probabilistic.txt"
report_path.write_text(report)
print(f"Report written to {report_path}")
print(report)